# Example 3.6.3 / 2: List the Constraints Still in the Problem

Companion notebook for **3.6.3 Slacks and Active Constraints** in the AMPL lab Documentation, section "List the Constraints Still in the Problem". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## List the Constraints Still in the Problem

"Active" also has a second meaning: a constraint that takes part in the problem sent to the
solver, as opposed to one removed by `drop` or by presolve. The method `astatus()` of an amplpy
constraint returns AMPL's status for it:

| Status | Meaning | Example on this page |
|---|---|---|
| `in` | in the problem sent to the solver | `Capacity` below |
| `drop` | removed by the `drop` command | `Mix` below |
| `pre` | eliminated by presolve, for example turned into a variable bound | `Floor` below |
| `fix` | a variable fixed by the `fix` command | `z` in the second example |
| `sub` | a defined variable, substituted out | `w` in the second example |
| `unused` | not used in the current problem | `u` in the second example |

The loop below walks every constraint with `get_constraints()` after `drop()` has removed `Mix`
and the model has been solved. It is adapted from the AMPL Colab notebook
[Identifying active constraints with Ampl](https://colab.ampl.com/notebooks/identifying-active-constraints-with-ampl.html):

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
var x >= 0;                        # Activity worth 3 per unit
var y >= 0;                        # Activity worth 2 per unit
maximize Value: 3 * x + 2 * y;     # Total value of both activities
subject to Capacity: x + y <= 10;  # Shared limit on x and y together
subject to Mix: x <= 6;            # Separate limit on x alone
subject to Floor: y >= 1;          # At least one unit of y

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
# This model has no external data.

In [5]:
# run.py
ampl.con["Mix"].drop()
ampl.solve(solver="highs", verbose=False)
print(ampl.solve_result)
for name, family in ampl.get_constraints():
    for index, con in family:
        print(name, con.astatus(), con.slack())

solved
Capacity in 0.0
Mix drop -3.0
Floor pre 0.0


### Pure AMPL: data.dat and model.run

Pure AMPL lists only the constraints still in the problem, through the generic names `_ncons`,
`_conname` and `_con` (Generic Names, Statistics and Message Limits (AMPL lab documentation, section 6.8.1)):

In [6]:
%%writefile data.dat
# This model has no external data.

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
drop Mix;
solve;
display {i in 1.._ncons} (_conname[i], _con[i].astatus, _con[i].slack);
display Mix.astatus;

HiGHS 1.14.0: optimal solution; objective 29
0 simplex iterations
0 barrier iterations
: _conname[i] _con[i].astatus _con[i].slack    :=
1   Capacity    in                   0
2   Floor       pre                  0
;

Mix.astatus = drop

